In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_groq import ChatGroq


In [8]:
llmGroq=ChatGroq(
  model="llama-3.3-70b-versatile",
  api_key=os.getenv("GROQ_API_KEY"),
  temperature=0.2,
)


# Ouput parser

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

json_prompt=PromptTemplate.from_template(
    "You are a helpful assistant. Please provide a JSON response with the following structure: {format_instructions}"
)

json_parser=SimpleJsonOutputParser()
json_chain=json_prompt | llmGroq | json_parser

In [14]:
res=json_chain.invoke({
  "format_instructions":"What is biggest country?"
})

In [15]:
res

{'question': 'What is biggest country?',
 'answer': 'Russia',
 'details': {'country': 'Russia',
  'land_area': '17,125,200 square kilometers',
  'percentage_of_world_land_area': '11%'}}

## Basic RAG implementation with LCEL

In [4]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load environment variables once
load_dotenv()

# 1. Load Document Data Loader
docs = PyPDFLoader("LangChain.pdf").load()

# 2. Split Document
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
  
# 3. Embed + Store (In-Memory)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings)

# 4. Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 5. Initialize LLM (API key is pulled automatically from .env)
llmGroq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
)

prompt=ChatPromptTemplate.from_template(
    "Answer using only this context:\n{context}\n\nQuestion: {question}"
)

# 6. Query and Retrieve
query = "Summarize the key point of my notes"
relevant_chunks = retriever.invoke(query)
context = "\n\n".join(d.page_content for d in relevant_chunks)
print("Context:\n", context)

# 7. Construct Prompt and Chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question using ONLY the provided context. If you do not know the answer based on the context, say 'I cannot find that in the notes.'"),
    ("user", "Context:\n{context}\n\nQuestion: {question}")
])
query="Summarize the key point of my notes"
chain = prompt | llmGroq | StrOutputParser()
response = chain.invoke({"context": context, "question": query})

print(response)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2617.52it/s]


Context:
 4 Vasilios Mavroudis
Memory: Enables applications to retain information from past interactions,
supporting both basic and advanced memory structures. This component is crit-
ical for maintaining context across sessions and delivering contextually aware
responses.
Indexes: Serve as structured databases that organize and store information,
allowing for efficient data retrieval when processing language queries.
Retrievers: Designed to work alongside indexes, retrievers fetch relevant data

4 Vasilios Mavroudis
Memory: Enables applications to retain information from past interactions,
supporting both basic and advanced memory structures. This component is crit-
ical for maintaining context across sessions and delivering contextually aware
responses.
Indexes: Serve as structured databases that organize and store information,
allowing for efficient data retrieval when processing language queries.
Retrievers: Designed to work alongside indexes, retrievers fetch relevant data

4 Vasi

# Temporary Memory (in-memory, session-only)

### RunnableWithMessageHistory is a LangChain wrapper class that automatically manages and injects conversation history into a chain.

In [6]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Simple in-memory store per session
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    llmGroq,
    get_session_history,
)

response = chain_with_memory.invoke(
    [{"role": "user", "content": "My name is Rahul"}],
    config={"configurable": {"session_id": "user123"}}
)

response2 = chain_with_memory.invoke(
    [{"role": "user", "content": "What's my name?"}],
    config={"configurable": {"session_id": "user123"}}
)
print(response)
print(response2.content) 

d:\Ai Engineer_\Week1\day6\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


content="Hello Rahul! It's nice to meet you. Is there something I can help you with or would you like to chat?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 39, 'total_tokens': 65, 'completion_time': 0.04359626, 'completion_tokens_details': None, 'prompt_time': 0.00347817, 'prompt_tokens_details': None, 'queue_time': 0.057307965, 'total_time': 0.04707443}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fa4e4-4b8e-76b3-8714-e5d4944dec55-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 26, 'total_tokens': 65}
Your name is Rahul.


# Permanent Memory

In [ ]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///chat_history.db"
    )

chain_with_memory = RunnableWithMessageHistory(
    llmGroq,
    get_session_history,
)